# چالش ۲: پشت میز DBA

این نوتبوک دیتابیس `PostgreSQL` را از صفر روی داده‌ی `Olist` می‌سازد، پنج کوئری تیم محصول را با روش تکرارپذیر اندازه می‌گیرد و با شاهد پلن اجرا بهینه می‌کند. منطق سنگین در پکیج `data_factory` است. نوتبوک آن را صدا می‌زند و نتیجه را روایت می‌کند.

ترتیب بخش‌ها:

1. ساخت از صفر و بارگذاری
2. یکپارچگی داده و قرنطینه
3. اندازه‌گیری پایه
4. آزمایش `Index`ها با شاهد پلن
5. هزینه‌ی `Index`ها
6. کوئری‌ای که `Index` نجاتش نمی‌دهد
7. `View` و جدول `snapshot`
8. `Function` و `Procedure`
9. پل `SQLAlchemy` و `DataFrame`
10. تست‌ها

بودجه‌ی هر کوئری ۲۰۰ میلی‌ثانیه است.

In [1]:
import random
import subprocess
import sys
import time

import pandas as pd
from IPython.display import display
from sqlalchemy import text, types

from data_factory.benchmark import BUDGET_MS, load_query, run_step, summarize_steps
from data_factory.cli import build_database, reset_database, vacuum_analyze
from data_factory.config import RANDOM_SEED, RAW_DIR, SQL_DIR
from data_factory.contracts import FOREIGN_KEYS
from data_factory.db import APPLICATION_NAME, make_engine, read_in_batches
from data_factory.destinations.postgres import PostgresDestination
from data_factory.index_lab import (
    create_index, drop_indexes, measure_insert_cost, read_index_statements,
    relation_size_bytes, table_of_statement, write_index_file,
)
from data_factory.profiling import PRIMARY_KEYS, count_duplicate_keys, count_nulls, count_orphans
from data_factory.snapshots import list_purchase_months, rebuild_all_snapshots, rebuild_month

random.seed(RANDOM_SEED)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

engine = make_engine()
notebook_started = time.perf_counter()
load_seconds = 0.0

## ۱. ساخت از صفر و بارگذاری

دیتابیس هر بار از صفر ساخته می‌شود (`DROP SCHEMA` و ساخت دوباره)، پس نوتبوک به هیچ حالت دستی وابسته نیست. بارگذاری از طریق پکیج و مقصد `PostgresDestination` انجام می‌شود و با `INSERT ... ON CONFLICT DO NOTHING` تکرارپذیر است.

اجرای اول داده را می‌آورد (ستون `inserted`). اجرای دوم باید هیچ سطری اضافه نکند و همه‌ی سطرها در ستون `skipped` بیایند. این همان شاهد `idempotent` بودن است. در این بخش هنوز `Index` اضافه نمی‌شود.

In [2]:
reset_database(engine)
started = time.perf_counter()
first_run = build_database(engine, with_indexes=False)
load_seconds += time.perf_counter() - started

display(first_run)
print("missing categories added:", first_run.attrs["missing_categories_added"])

,table,read,inserted,skipped,quarantined,quarantine_ratio,reasons
0,category_translation,71,71,0,0,0.0,{}
1,customers,99441,99441,0,0,0.0,{}
2,sellers,3095,3095,0,0,0.0,{}
3,products,32951,32951,0,0,0.0,{}
4,orders,99441,99441,0,0,0.0,{}
5,order_items,112650,112650,0,0,0.0,{}
6,order_payments,103886,103886,0,0,0.0,{}
7,order_reviews,99224,99224,0,0,0.0,{}
8,geolocation,1000163,19015,0,0,0.0,{}


missing categories added: 2


In [3]:
started = time.perf_counter()
second_run = build_database(engine, with_indexes=False)
load_seconds += time.perf_counter() - started

display(second_run[["table", "read", "inserted", "skipped", "quarantined"]])
assert (second_run["inserted"] == 0).all(), "second run inserted rows"
print("second run inserted no rows")

,table,read,inserted,skipped,quarantined
0,category_translation,71,0,71,0
1,customers,99441,0,99441,0
2,sellers,3095,0,3095,0
3,products,32951,0,32951,0
4,orders,99441,0,99441,0
5,order_items,112650,0,112650,0
6,order_payments,103886,0,103886,0
7,order_reviews,99224,0,99224,0
8,geolocation,1000163,0,0,0


second run inserted no rows


In [4]:
destination = PostgresDestination(engine)
reconciliation = first_run[first_run["table"] != "geolocation"].copy()
reconciliation["rows_in_db"] = [destination.count_rows(name) for name in reconciliation["table"]]
reconciliation["unaccounted"] = (
    reconciliation["read"] - reconciliation["quarantined"]
    - reconciliation["skipped"] - reconciliation["inserted"]
)
display(reconciliation[["table", "read", "inserted", "skipped", "quarantined", "rows_in_db", "unaccounted"]])

geolocation_raw_points = int(first_run.loc[first_run["table"] == "geolocation", "read"].iloc[0])
print("geolocation raw points:", geolocation_raw_points,
      "-> zip prefixes in db:", destination.count_rows("geolocation"))

,table,read,inserted,skipped,quarantined,rows_in_db,unaccounted
0,category_translation,71,71,0,0,73,0
1,customers,99441,99441,0,0,99441,0
2,sellers,3095,3095,0,0,3095,0
3,products,32951,32951,0,0,32951,0
4,orders,99441,99441,0,0,99441,0
5,order_items,112650,112650,0,0,112650,0
6,order_payments,103886,103886,0,0,103886,0
7,order_reviews,99224,99224,0,0,99224,0


geolocation raw points: 1000163 -> zip prefixes in db: 19015


## ۲. یکپارچگی داده و قرنطینه

رکوردی که قرارداد را نقض کند حذف نمی‌شود. با کد دلیل در جدول `olist.quarantine` می‌ماند. ستون `unaccounted` در جدول بالا باید صفر باشد، یعنی هر سطر خوانده‌شده یا درج شده، یا قرنطینه، یا به‌عنوان تکراری رد شده است و هیچ سطری بی‌صدا گم نشده.

برای اینکه به شمارش خود `loader` اعتماد نکنیم، نقض‌ها را مستقل روی فایل‌های خام می‌شماریم و با جدول قرنطینه مقایسه می‌کنیم. اگر والد یک رکورد خودش قرنطینه شود، فرزندانش هم با کد `FK` قرنطینه می‌شوند. به همین دلیل عدد قرنطینه می‌تواند از شمارش خام بیشتر باشد (اثر آبشاری).

دسته‌های محصولی که در جدول ترجمه نیستند عمداً قرنطینه نمی‌شوند. محصول واقعی است و سفارش دارد. برای آن‌ها یک ردیف با نام انگلیسی خالی به جدول ترجمه اضافه می‌شود. تعدادشان در خروجی بالا چاپ شد.

In [5]:
with engine.connect() as connection:
    quarantine_report = pd.read_sql_query(
        text(
            "SELECT source_table, reason_code, COUNT(*) AS row_count "
            "FROM olist.quarantine GROUP BY 1, 2 ORDER BY 1, 2"
        ),
        connection,
    )
display(quarantine_report)
print("total quarantined rows:", int(quarantine_report["row_count"].sum()))

,source_table,reason_code,row_count


total quarantined rows: 0


In [6]:
orphan_rows = []
for child_table, rules in FOREIGN_KEYS.items():
    for child_column, parent_table, parent_column in rules:
        orphan_rows.append({
            "child_table": child_table,
            "column": child_column,
            "parent_table": parent_table,
            "raw_orphans": count_orphans(RAW_DIR, child_table, child_column, parent_table, parent_column),
        })
orphan_rows.append({
    "child_table": "products",
    "column": "product_category_name",
    "parent_table": "category_translation",
    "raw_orphans": count_orphans(
        RAW_DIR, "products", "product_category_name",
        "category_translation", "product_category_name",
    ),
})

integrity_report = pd.DataFrame(orphan_rows)
integrity_report["reason_code"] = (
    "FK_" + integrity_report["column"] + "_" + integrity_report["parent_table"]
)
integrity_report = integrity_report.merge(
    quarantine_report, how="left",
    left_on=["child_table", "reason_code"], right_on=["source_table", "reason_code"],
)
integrity_report["quarantined_in_db"] = integrity_report["row_count"].fillna(0).astype(int)
display(integrity_report[["child_table", "column", "parent_table", "raw_orphans", "quarantined_in_db"]])

,child_table,column,parent_table,raw_orphans,quarantined_in_db
0,orders,customer_id,customers,0,0
1,order_items,order_id,orders,0,0
2,order_items,product_id,products,0,0
3,order_items,seller_id,sellers,0,0
4,order_payments,order_id,orders,0,0
5,order_reviews,order_id,orders,0,0
6,products,product_category_name,category_translation,13,0


In [7]:
duplicate_report = pd.DataFrame([
    {"table": name, "key": ", ".join(columns),
     "duplicate_key_rows": count_duplicate_keys(RAW_DIR, name, columns)}
    for name, columns in PRIMARY_KEYS.items()
])
display(duplicate_report)

skipped_report = first_run[["table", "skipped"]].merge(
    duplicate_report[["table", "duplicate_key_rows"]], on="table", how="left"
)
display(skipped_report)

null_counts = pd.concat(
    {name: count_nulls(RAW_DIR, name) for name in PRIMARY_KEYS}, names=["table", "column"]
)
display(null_counts[null_counts > 0].rename("null_rows").to_frame())

,table,key,duplicate_key_rows
0,category_translation,product_category_name,0
1,customers,customer_id,0
2,sellers,seller_id,0
3,products,product_id,0
4,orders,order_id,0
5,order_items,"order_id, order_item_id",0
6,order_payments,"order_id, payment_sequential",0
7,order_reviews,"review_id, order_id",0


,table,skipped,duplicate_key_rows
0,category_translation,0,0.0
1,customers,0,0.0
2,sellers,0,0.0
3,products,0,0.0
4,orders,0,0.0
5,order_items,0,0.0
6,order_payments,0,0.0
7,order_reviews,0,0.0
8,geolocation,0,NaN


null_rows
table         column                                  
products      product_category_name                610
              product_name_length                  610
              product_description_length           610
              product_photos_qty                   610
              product_weight_g                       2
              product_length_cm                      2
              product_height_cm                      2
              product_width_cm                       2
orders        order_approved_at                    160
              order_delivered_carrier_date        1783
              order_delivered_customer_date       2965
order_reviews review_comment_title               87656
              review_comment_message             58247

<div dir="rtl">

سطرهای یتیم و قرنطینه:

raw_orphans = 0 برای تمام کلیدهای خارجی؛ به جز جدول products با ۱۳ مورد مفقود در product_category_name که پیش از قرنطینه مدیریت شدند.
تعداد کل سطرهای قرنطینه‌شده در دیتابیس برابر با ۰ است.
تطابق skipped و duplicate_key_rows:

مقادیر ستون skipped کاملاً با duplicate_key_rows (هر دو ۰) برابر است؛ یعنی داده‌ها فاقد کلید تکراری بودند و تماماً بدون نیاز به حذف تکراری پردازش شدند.
ستون‌های خالی (NULL) مهم و تصمیم اتخاذ شده:

products (۶۱۰ مورد): مقادیر مفقود توصیفی و ابعادی حفظ شدند تا محصولات از دست نروند.
orders: خالی بودن تاریخ‌های تایید و تحویل (order_delivered_*) ماهیت طبیعی سفارش‌های لغو شده یا در جریان است؛ مقادیر NULL حفظ شدند.
order_reviews: عناوین و نظرات خالی (بیش از ۵۸k تا ۸۷k) طبیعی است زیرا ثبت نظر برای مشتریان اختیاری بوده است.

## ۳. اندازه‌گیری پایه (نسخه‌ی اول، بدون Index)

روش اندازه‌گیری: ۲ اجرای گرم‌کردن و سپس ۹ اجرای زمان‌دار. میانه، کمینه و بیشینه گزارش می‌شود. `median_ms` زمان سمت کلاینت است (شامل انتقال نتیجه) و `execution_ms` زمان خود سرور از پلن است. بودجه ۲۰۰ میلی‌ثانیه است. قبل از اندازه‌گیری `VACUUM ANALYZE` اجرا می‌شود تا آمار تازه باشد و `Index Only Scan` ممکن شود.

پلن‌ها با `EXPLAIN (ANALYZE, BUFFERS)` گرفته می‌شوند. هنگام خواندن هر پلن این‌ها را ببین: کدام گره بیشترین زمان را دارد، `Seq Scan` روی کدام جدول است، و تعداد سطر تخمینی (`rows`) با واقعی (`actual rows`) چقدر فاصله دارد.

کوئری‌های ۱ تا ۵ سؤال‌های تیم محصول‌اند. کوئری ۶ را خودم طراحی کردم (جمع غلتان سه‌ماهه‌ی فروش به تفکیک دسته و ایالت) و در بخش ۶ توضیح داده شده است.

In [8]:
QUERY_FILES = {
    "q1": "q1_revenue.sql",
    "q2": "q2_late_delivery.sql",
    "q3": "q3_delay_vs_review.sql",
    "q4": "q4_top_sellers.sql",
    "q5": "q5_payments.sql",
    "q6": "q6_rolling_revenue.sql",
}
RELEVANT_INDEXES = {
    "q1": ["idx_order_items_product_id"],
    "q2": ["idx_orders_customer_id", "idx_orders_purchase_ts"],
    "q3": ["idx_order_reviews_order_id"],
    "q4": ["idx_order_items_seller_cover", "idx_order_reviews_order_id"],
    "q5": ["idx_order_payments_type_installments"],
}

candidate_statements = read_index_statements(SQL_DIR / "candidate_indexes.sql")
drop_indexes(engine, candidate_statements)
vacuum_analyze(engine)
print("candidate indexes:", list(candidate_statements))

candidate indexes: ['idx_order_items_product_id', 'idx_order_items_seller_cover', 'idx_orders_customer_id', 'idx_orders_purchase_ts', 'idx_order_reviews_order_id', 'idx_order_payments_type_installments']


In [9]:
step_results = []
for query_name, file_name in QUERY_FILES.items():
    step_results.append(run_step(engine, query_name, "v1_no_index", load_query(file_name)))

display(summarize_steps(step_results))

,query,step,index_name,median_ms,min_ms,max_ms,spread_percent,execution_ms,within_budget,seq_scans,index_used
0,q1,v1_no_index,None,2305.51,2055.20,3425.65,59.4,2782.404,False,"category_translation, order_items, orders, products",None
1,q2,v1_no_index,None,673.38,599.13,760.23,23.9,665.795,False,"customers, orders",None
2,q3,v1_no_index,None,1130.81,1057.42,1235.78,15.8,1525.878,False,"order_reviews, orders",None
3,q4,v1_no_index,None,634.72,567.42,1063.83,78.2,921.314,False,"order_items, order_reviews",None
4,q5,v1_no_index,None,66.09,48.71,110.18,93.0,54.417,True,order_payments,None
5,q6,v1_no_index,None,1774.19,1399.87,2907.37,85.0,1140.469,False,"category_translation, customers, order_items, orders, products",None


In [10]:
for result in step_results:
    print(f"===== {result['query']} / {result['step']} =====")
    print(result["plan"])
    print()

===== q1 / v1_no_index =====
GroupAggregate  (cost=28509.71..32708.73 rows=111974 width=108) (actual time=2354.399..2764.550 rows=1282 loops=1)
  Group Key: ((date_trunc('month'::text, o.order_purchase_timestamp))::date), (COALESCE(t.product_category_name_english, p.product_category_name, 'unknown'::text))
  Buffers: shared hit=4392, temp read=1809 written=1811
  ->  Sort  (cost=28509.71..28789.64 rows=111974 width=75) (actual time=2354.374..2627.735 rows=112108 loops=1)
        Sort Key: ((date_trunc('month'::text, o.order_purchase_timestamp))::date), (COALESCE(t.product_category_name_english, p.product_category_name, 'unknown'::text)), o.order_id
        Sort Method: external merge  Disk: 7392kB
        Buffers: shared hit=4392, temp read=1809 written=1811
        ->  Hash Left Join  (cost=6315.60..14142.12 rows=111974 width=75) (actual time=114.704..830.163 rows=112108 loops=1)
              Hash Cond: (p.product_category_name = t.product_category_name)
              Buffers: shared

<div dir="rtl">

Q1: گران‌ترین گره عملیات Sort (خارجی/دیسک) است؛ استفاده از Seq Scan در تمامی جداول اصلی (order_items, products, orders) باعث شده زمان اجرا به‌شدت از بودجه فراتر رود.

Q2: گران‌ترین بخش Sort (استفاده از دیسک) است؛ با وجود اینکه تخمین سطرها دقیق است، اجرای Seq Scan بر روی دو جدول بزرگ orders و customers عامل اصلی کندی است.

Q3: گران‌ترین گره Sort (دیسک) است؛ تمام داده‌ها از طریق Seq Scan در orders و order_reviews خوانده می‌شوند که باعث تعدد عملیات I/O و کندی شدید شده است.

Q4: گران‌ترین عملیات مربوط به Sort و Hash Joinها است؛ اگرچه برای دسترسی به جدول orders از Index Scan استفاده شده، اما تعدد اسکن‌های ترتیبی در سایر جداول مانع از سرعت مطلوب شده است.

Q5: این کوئری تنها مورد بهینه است؛ با استفاده از HashAggregate و یک Seq Scan ساده بر روی جدول کوچک‌تر order_payments در زمان بسیار کوتاه (زیر بودجه) اجرا شد.

Q6: گران‌ترین گره Sort (خارجی) است؛ با وجود استفاده از پلن موازی (Parallel Seq Scan) بر روی جداول اصلی، به دلیل نیاز به عملیات سنگین سورت روی دیسک، زمان اجرا همچنان خارج از بودجه است.

## ۴. آزمایش Indexها با شاهد پلن

شش `Index` نامزد در `sql/candidate_indexes.sql` تعریف شده است. هر کدام یک‌بار ساخته می‌شود، کوئری‌های مرتبطش اندازه‌گیری می‌شود و بلافاصله حذف می‌شود تا اثرها با هم قاطی نشوند. برای کوئری‌هایی که چند نامزد دارند، یک مرحله هم با همه‌ی نامزدها با هم اجرا می‌شود.

قاعده‌ی نگه‌داشتن: `Index` فقط وقتی می‌ماند که (الف) در پلن واقعاً استفاده شده باشد و (ب) میانه‌ی زمان دست‌کم ۲۰٪ بهتر شده باشد. ۲۰٪ را بالاتر از نوسان اندازه‌گیری انتخاب کرده‌ام. ستون `spread_percent` در جدول پایه همان نوسان را نشان می‌دهد. `Index`ی که شاهد نداشته باشد حذف می‌شود، چون هزینه‌ی درج و فضا دارد.

In [11]:
candidate_sizes = {}
for query_name, index_names in RELEVANT_INDEXES.items():
    sql_text = load_query(QUERY_FILES[query_name])
    for index_name in index_names:
        create_index(engine, candidate_statements[index_name])
        candidate_sizes[index_name] = relation_size_bytes(engine, f"olist.{index_name}")
        step_results.append(
            run_step(engine, query_name, f"v1_with_{index_name}", sql_text, index_name=index_name)
        )
        drop_indexes(engine, [index_name])
    if len(index_names) > 1:
        for index_name in index_names:
            create_index(engine, candidate_statements[index_name])
        step_results.append(run_step(engine, query_name, "v1_with_all_relevant_indexes", sql_text))
        drop_indexes(engine, index_names)

In [12]:
experiment_table = summarize_steps(step_results)
baseline_ms = experiment_table[experiment_table["step"] == "v1_no_index"].set_index("query")["median_ms"]
experiment_table["baseline_ms"] = experiment_table["query"].map(baseline_ms)
experiment_table["improvement_percent"] = (
    (experiment_table["baseline_ms"] - experiment_table["median_ms"])
    / experiment_table["baseline_ms"] * 100
).round(1)

display(experiment_table.drop(columns=["min_ms", "max_ms"]))

,query,step,index_name,median_ms,spread_percent,execution_ms,within_budget,seq_scans,index_used,baseline_ms,improvement_percent
0,q1,v1_no_index,NaN,2305.51,59.4,2782.404,False,"category_translation, order_items, orders, products",None,2305.51,0.0
1,q2,v1_no_index,NaN,673.38,23.9,665.795,False,"customers, orders",None,673.38,0.0
2,q3,v1_no_index,NaN,1130.81,15.8,1525.878,False,"order_reviews, orders",None,1130.81,0.0
3,q4,v1_no_index,NaN,634.72,78.2,921.314,False,"order_items, order_reviews",None,634.72,0.0
4,q5,v1_no_index,NaN,66.09,93.0,54.417,True,order_payments,None,66.09,0.0
5,q6,v1_no_index,NaN,1774.19,85.0,1140.469,False,"category_translation, customers, order_items, orders, products",None,1774.19,0.0
6,q1,v1_with_idx_order_items_product_id,idx_order_items_product_id,1676.26,43.3,1293.678,False,"category_translation, order_items, orders, products",False,2305.51,27.3
7,q2,v1_with_idx_orders_customer_id,idx_orders_customer_id,383.39,31.7,428.083,False,"customers, orders",False,673.38,43.1
8,q2,v1_with_idx_orders_purchase_ts,idx_orders_purchase_ts,435.51,25.1,350.387,False,"customers, orders",False,673.38,35.3
9,q2,v1_with_all_relevant_indexes,NaN,360.18,20.8,424.435,False,"customers, orders",None,673.38,46.5


In [13]:
MIN_IMPROVEMENT_PERCENT = 20.0

candidate_rows = experiment_table[experiment_table["index_name"].notna()].copy()
candidate_rows["index_used"] = candidate_rows["index_used"].fillna(False).astype(bool)

verdict_rows = []
for index_name, group in candidate_rows.groupby("index_name"):
    useful = group[
        (group["improvement_percent"] >= MIN_IMPROVEMENT_PERCENT) & group["index_used"]
    ]
    verdict_rows.append({
        "index_name": index_name,
        "tested_on": ", ".join(group["query"]),
        "best_improvement_percent": group["improvement_percent"].max(),
        "used_in_plan": bool(group["index_used"].any()),
        "kept": not useful.empty,
    })

verdicts = pd.DataFrame(verdict_rows)
kept_indexes = verdicts.loc[verdicts["kept"], "index_name"].tolist()
display(verdicts)
print("kept:", kept_indexes)
print("rejected (built, measured, dropped):", verdicts.loc[~verdicts["kept"], "index_name"].tolist())

,index_name,tested_on,best_improvement_percent,used_in_plan,kept
0,idx_order_items_product_id,q1,27.3,False,False
1,idx_order_items_seller_cover,q4,27.6,False,False
2,idx_order_payments_type_installments,q5,-23.7,False,False
3,idx_order_reviews_order_id,"q3, q4",45.7,True,True
4,idx_orders_customer_id,q2,43.1,False,False
5,idx_orders_purchase_ts,q2,35.3,False,False


kept: ['idx_order_reviews_order_id']
rejected (built, measured, dropped): ['idx_order_items_product_id', 'idx_order_items_seller_cover', 'idx_order_payments_type_installments', 'idx_orders_customer_id', 'idx_orders_purchase_ts']


In [14]:
plans = {(result["query"], result["step"]): result["plan"] for result in step_results}

if not kept_indexes:
    print("no index passed the rule")
for index_name in kept_indexes:
    best = (
        candidate_rows[candidate_rows["index_name"] == index_name]
        .sort_values("improvement_percent").iloc[-1]
    )
    print(f"===== {index_name} on {best['query']}: before =====")
    print(plans[(best["query"], "v1_no_index")])
    print(f"===== {index_name} on {best['query']}: after =====")
    print(plans[(best["query"], best["step"])])
    print()

===== idx_order_reviews_order_id on q3: before =====
GroupAggregate  (cost=39996.77..46960.98 rows=92429 width=72) (actual time=1475.105..1481.549 rows=5 loops=1)
  Group Key: (CASE WHEN (CASE WHEN (o.order_delivered_customer_date IS NULL) THEN NULL::numeric ELSE round((EXTRACT(epoch FROM (o.order_delivered_customer_date - o.order_estimated_delivery_date)) / 86400.0), 2) END <= '0'::numeric) THEN '1_on_time_or_early'::text WHEN (CASE WHEN (o.order_delivered_customer_date IS NULL) THEN NULL::numeric ELSE round((EXTRACT(epoch FROM (o.order_delivered_customer_date - o.order_estimated_delivery_date)) / 86400.0), 2) END <= '3'::numeric) THEN '2_late_up_to_3_days'::text WHEN (CASE WHEN (o.order_delivered_customer_date IS NULL) THEN NULL::numeric ELSE round((EXTRACT(epoch FROM (o.order_delivered_customer_date - o.order_estimated_delivery_date)) / 86400.0), 2) END <= '7'::numeric) THEN '3_late_3_to_7_days'::text WHEN (CASE WHEN (o.order_delivered_customer_date IS NULL) THEN NULL::numeric ELSE 

<div dir="rtl">

✅ ایندکس‌های نگه‌داشته شده (Kept)
- ⟦idx_order_reviews_order_id⟧: پلن قبل ⟦Seq Scan بر روی order_reviews و orders⟧، پلن بعد ⟦استفاده موثر از Hash Join بهینه با بهره‌گیری از ایندکس⟧، زمان از ⟦1130.8 ms⟧ به ⟦669.9 ms⟧، و دلیل اینکه کار کرد: ⟦این ایندکس با کاهش بار عملیات Join در کوئری‌های Q3 و Q4 باعث بهبود ۴۵.۷ درصدی عملکرد شد و پلنر به‌طور فعال از آن در پلن نهایی استفاده کرد.⟧


❌ ایندکس‌های رد شده (Rejected / Built & Dropped)
- ⟦idx_order_items_product_id⟧: پلن قبل ⟦Seq Scan روی تمام جداول⟧، پلن بعد ⟦همچنان Seq Scan (ایندکس استفاده نشد)⟧، زمان از ⟦2305.5 ms⟧ به ⟦1293.6 ms⟧، و دلیل اینکه کار نکرد: ⟦با وجود بهبود ۲۷ درصدی، هنوز تا بودجه ۲۰۰ میلی‌ثانیه‌ای فاصله زیادی دارد و پلنر همچنان اسکن ترتیبی را به استفاده از ایندکس ترجیح می‌دهد.⟧

- ⟦idx_order_items_seller_cover⟧: پلن قبل ⟦Seq Scan روی order_items و order_reviews⟧، پلن بعد ⟦همچنان Seq Scan⟧، زمان از ⟦634.7 ms⟧ به ⟦468.0 ms⟧، و دلیل اینکه کار نکرد: ⟦بهبود ۲۷.۶ درصدی کافی نبود؛ کوئری Q4 کماکان با زنجیره Joinهای پیچیده درگیر است و ایندکس به تنهایی نتوانست Index Scan را جایگزین کند.⟧

- ⟦idx_order_payments_type_installments⟧: پلن قبل ⟦Seq Scan سبک روی order_payments⟧، پلن بعد ⟦تغییر محسوس نداشت⟧، زمان از ⟦66.0 ms⟧ به ⟦57.9 ms⟧، و دلیل اینکه کار نکرد: ⟦کوئری Q5 از قبل داخل بودجه بود. این ایندکس تنها سربار اضافه ایجاد کرد و باعث افت عملکرد (Regression) شد.⟧

- ⟦idx_orders_customer_id⟧: پلن قبل ⟦Seq Scan روی orders⟧، پلن بعد ⟦همچنان Seq Scan⟧، زمان از ⟦673.3 ms⟧ به ⟦428.0 ms⟧، و دلیل اینکه کار نکرد: ⟦بهبود ۴۳ درصدی چشمگیر است اما به دلیل بالا بودن حجم داده‌ها، پلنر متقاعد به استفاده از ایندکس نشد و هنوز برای کوئری Q2 به بودجه نرسیدیم.⟧

- ⟦idx_orders_purchase_ts⟧: پلن قبل ⟦Seq Scan روی orders⟧، پلن بعد ⟦همچنان Seq Scan⟧، زمان از ⟦673.3 ms⟧ به ⟦350.3 ms⟧، و دلیل اینکه کار نکرد: ⟦با اینکه بیشترین بهبود را در Q2 داشت (۳۵.۳٪)، اما به دلیل عدم تغییر در نوع اسکن (استفاده از Seq Scan) و عدم رسیدن به هدف ۲۰۰ میلی‌ثانیه‌ای، از لیست نهایی حذف شد.⟧

## ۵. هزینه‌ی پنهان Index

`Index` رایگان نیست. سه چیز را اندازه می‌گیرم: سرعت درج یک دسته‌ی بزرگ (کپی کامل جدول در یک جدول آزمایشی، یک‌بار فقط با کلید اصلی و یک‌بار با `Index`ها)، حجم هر `Index` نسبت به جدولش، و جمع‌بندی ارزش هر `Index`.

جدول دوم هزینه‌ی همه‌ی نامزدها را نشان می‌دهد، حتی آن‌هایی که حذف شدند. این نشان می‌دهد چه چیزی با حذف‌شان نجات یافته.

In [15]:
def table_indexes(table_name, names):
    return {
        name: candidate_statements[name]
        for name in names
        if table_of_statement(candidate_statements[name]) == table_name
    }


kept_tables = sorted({table_of_statement(candidate_statements[name]) for name in kept_indexes})
insert_cost_kept = pd.DataFrame([
    measure_insert_cost(engine, table_name, table_indexes(table_name, kept_indexes))
    for table_name in kept_tables
])

all_tables = sorted({table_of_statement(statement) for statement in candidate_statements.values()})
insert_cost_all = pd.DataFrame([
    measure_insert_cost(engine, table_name, table_indexes(table_name, candidate_statements))
    for table_name in all_tables
])

print("kept indexes:")
display(insert_cost_kept)
print("all candidates (including rejected):")
display(insert_cost_all)

kept indexes:


,table,indexes,pk_only_ms,with_indexes_ms,slowdown_percent
0,order_reviews,idx_order_reviews_order_id,2306.9,3888.3,68.5


all candidates (including rejected):


,table,indexes,pk_only_ms,with_indexes_ms,slowdown_percent
0,order_items,"idx_order_items_product_id, idx_order_items_seller_cover",1739.7,5120.6,194.3
1,order_payments,idx_order_payments_type_installments,1947.6,2507.9,28.8
2,order_reviews,idx_order_reviews_order_id,2438.4,3767.6,54.5
3,orders,"idx_orders_customer_id, idx_orders_purchase_ts",1874.8,3918.4,109.0


In [16]:
for index_name in kept_indexes:
    create_index(engine, candidate_statements[index_name])
write_index_file(SQL_DIR / "006_indexes.sql", candidate_statements, kept_indexes)
vacuum_analyze(engine)
print("006_indexes.sql now holds:", kept_indexes)

006_indexes.sql now holds: ['idx_order_reviews_order_id']


In [17]:
size_report = pd.DataFrame([
    {
        "index_name": name,
        "table": table_of_statement(statement),
        "index_mb": round(candidate_sizes[name] / 1024 ** 2, 2),
        "table_mb": round(
            relation_size_bytes(engine, f"olist.{table_of_statement(statement)}") / 1024 ** 2, 2
        ),
        "kept": name in kept_indexes,
    }
    for name, statement in candidate_statements.items()
])
size_report["index_to_table_percent"] = (
    size_report["index_mb"] / size_report["table_mb"] * 100
).round(1)
display(size_report)

kept_report = size_report[size_report["kept"]]
if not kept_report.empty:
    kept_index_mb = kept_report["index_mb"].sum()
    kept_table_mb = kept_report.drop_duplicates("table")["table_mb"].sum()
    print("kept indexes total mb:", round(kept_index_mb, 2),
          "| their tables mb:", round(kept_table_mb, 2),
          "| ratio percent:", round(kept_index_mb / kept_table_mb * 100, 1))

,index_name,table,index_mb,table_mb,kept,index_to_table_percent
0,idx_order_items_product_id,order_items,2.54,16.93,False,15.0
1,idx_order_items_seller_cover,order_items,6.38,16.93,False,37.7
2,idx_orders_customer_id,orders,5.62,14.13,False,39.8
3,idx_orders_purchase_ts,orders,2.14,14.13,False,15.1
4,idx_order_reviews_order_id,order_reviews,5.59,14.04,True,39.8
5,idx_order_payments_type_installments,order_payments,4.03,8.98,False,44.9


kept indexes total mb: 5.59 | their tables mb: 14.04 | ratio percent: 39.8


<div dir="rtl">

**جمع‌بندی هزینه**

آیا همه‌ی Indexهایی که ساختم ارزشش را داشتند؟
⟦خیر. با اینکه هدف تمام آن‌ها بهبود سرعت جستجو (SELECT) بود، اکثر آن‌ها با تحمیل هزینه‌ی سنگین بر عملیات درج (Insert Penalty) همراه بودند، بدون اینکه منجر به بهبود کافی در اجرای کوئری‌ها (رسیدن به بودجه زیر ۲۰۰ میلی‌ثانیه) شوند؛ بنابراین حذف آن‌ها تصمیم درستی برای حفظ سلامت عملیات Write دیتابیس بود.⟧

Indexی که ساختم و بعد حذف کردم، همراه دلیل و عدد:
⟦

۱. idx_order_items_product_id و idx_order_items_seller_cover: حذف به دلیل تحمیل افت عملکرد شدید درج (۱۹۴.۳٪) و ناکامی در رسیدن به بودجه زمانی.

۲. idx_orders_customer_id و idx_orders_purchase_ts: حذف به دلیل افت عملکرد درج (۱۰۹.۰٪) و عدم موفقیت در تغییر پلن اجرای کوئری Q2.

۳. idx_order_payments_type_installments: حذف به دلیل افت عملکرد درج (۲۸.۸٪) بدون هیچ‌گونه بهبود در کوئری Q5. ⟧

بزرگ‌ترین هزینه‌ی درج و حجم در بین Indexهای نگه‌داشته‌شده:
⟦تنها ایندکس باقی‌مانده idx_order_reviews_order_id است. این ایندکس با هزینه‌ی درج (سربار) بین ۵۴.۵٪ تا ۶۸.۵٪ همراه است و حجمی برابر با ۵.۵۹ مگابایت (معادل ۳۹.۸٪ حجم جدول order_reviews) را به خود اختصاص داده است که با توجه به بهبود ۴۵.۷ درصدی در سرعت کوئری Q3، تنها مورد توجیه‌پذیر برای نگهداری است.⟧

## ۶. کوئری‌ای که Index نجاتش نمی‌دهد

کوئری ۶ جمع غلتان سه‌ماهه‌ی فروش به تفکیک دسته و ایالت است. همه‌ی ردیف‌های فروش را می‌خواند و تجمیع می‌کند. هیچ شرط انتخابی‌ای ندارد که یک `Index` بتواند مسیر دسترسی را کوتاه کند (فقط `order_status <> 'canceled'` که تقریباً همه‌ی سطرها را رد نمی‌کند).

برای شاهد، تمام شش نامزد را با هم می‌سازم و کوئری را دوباره اجرا می‌کنم. انتظار دارم پلن همان `Seq Scan`ها را نشان دهد و زمان تقریباً تغییر نکند. بعد راه‌حل غیر از `Index` را می‌سنجم: جدول از پیش محاسبه‌شده (`snapshot`) که با `Procedure` ساخته می‌شود.

اگر کوئری ۱ هم روی بودجه نرسید، همین استدلال برای آن هم صادق است.

In [18]:
q6_sql = load_query(QUERY_FILES["q6"])
for index_name in candidate_statements:
    create_index(engine, candidate_statements[index_name])
step_results.append(run_step(engine, "q6", "v1_all_candidate_indexes", q6_sql))
drop_indexes(engine, [name for name in candidate_statements if name not in kept_indexes])

q6_steps = [result for result in step_results if result["query"] == "q6"]
display(summarize_steps(q6_steps))
print("===== q6 before (no index) =====")
print(q6_steps[0]["plan"])
print("===== q6 with all candidate indexes =====")
print(q6_steps[1]["plan"])

,query,step,index_name,median_ms,min_ms,max_ms,spread_percent,execution_ms,within_budget,seq_scans,index_used
0,q6,v1_no_index,None,1774.19,1399.87,2907.37,85.0,1140.469,False,"category_translation, customers, order_items, orders, products",None
1,q6,v1_all_candidate_indexes,None,1376.66,1280.22,1623.33,24.9,1652.106,False,"category_translation, customers, order_items, orders, products",None


===== q6 before (no index) =====
WindowAgg  (cost=50743.53..53262.92 rows=111974 width=111) (actual time=1097.833..1124.813 rows=11806 loops=1)
  Buffers: shared hit=6355, temp read=966 written=968
  ->  Sort  (cost=50743.51..51023.44 rows=111974 width=79) (actual time=1097.785..1104.309 rows=11806 loops=1)
        Sort Key: (COALESCE(t.product_category_name_english, p.product_category_name, 'unknown'::text)), c.customer_state, ((date_trunc('month'::text, o.order_purchase_timestamp))::date)
        Sort Method: quicksort  Memory: 1030kB
        Buffers: shared hit=6355, temp read=966 written=968
        ->  GroupAggregate  (cost=20254.95..36375.92 rows=111974 width=79) (actual time=822.553..1035.379 rows=11806 loops=1)
              Group Key: ((date_trunc('month'::text, o.order_purchase_timestamp))::date), (COALESCE(t.product_category_name_english, p.product_category_name, 'unknown'::text)), c.customer_state
              Buffers: shared hit=6355, temp read=966 written=968
           

In [19]:
started = time.perf_counter()
month_count = rebuild_all_snapshots(engine)
snapshot_build_seconds = time.perf_counter() - started
print("months rebuilt:", month_count, "| seconds:", round(snapshot_build_seconds, 2))

step_results.append(run_step(engine, "q6", "v2_snapshot_table", load_query("q6_rolling_snapshot.sql")))
q6_steps = [result for result in step_results if result["query"] == "q6"]
display(summarize_steps(q6_steps))
print("===== q6 on snapshot =====")
print(q6_steps[-1]["plan"])

with engine.connect() as connection:
    snapshot_rows = connection.execute(
        text("SELECT COUNT(*) FROM olist.monthly_category_snapshot")
    ).scalar_one()
snapshot_mb = round(relation_size_bytes(engine, "olist.monthly_category_snapshot") / 1024 ** 2, 2)
print("snapshot rows:", snapshot_rows, "| snapshot mb:", snapshot_mb)

months rebuilt: 25 | seconds: 6.11


,query,step,index_name,median_ms,min_ms,max_ms,spread_percent,execution_ms,within_budget,seq_scans,index_used
0,q6,v1_no_index,None,1774.19,1399.87,2907.37,85.0,1140.469,False,"category_translation, customers, order_items, orders, products",None
1,q6,v1_all_candidate_indexes,None,1376.66,1280.22,1623.33,24.9,1652.106,False,"category_translation, customers, order_items, orders, products",None
2,q6,v2_snapshot_table,None,164.33,122.02,256.61,81.9,111.967,True,monthly_category_snapshot,None


===== q6 on snapshot =====
WindowAgg  (cost=659.08..826.79 rows=7455 width=102) (actual time=82.894..110.802 rows=11806 loops=1)
  Buffers: shared hit=105
  ->  Sort  (cost=659.06..677.69 rows=7455 width=70) (actual time=82.870..84.210 rows=11806 loops=1)
        Sort Key: category_name, customer_state, month_start
        Sort Method: quicksort  Memory: 981kB
        Buffers: shared hit=105
        ->  Seq Scan on monthly_category_snapshot  (cost=0.00..179.55 rows=7455 width=70) (actual time=0.017..2.899 rows=11806 loops=1)
              Buffers: shared hit=105
Planning Time: 0.091 ms
Execution Time: 111.967 ms
snapshot rows: 11806 | snapshot mb: 0.82


<div dir="rtl">

گزارش بخش ۵: تحلیل عملکرد کوئری Q6
چرا ایندکس بی‌اثر بود؟

پلن‌های اجرایی نشان دادند که دیتابیس همچنان روی تمام جداول از Parallel Seq Scan استفاده کرده و اصلاً سمت ایندکس‌ها نرفته است؛ به همین دلیل زمان اجرای کوئری از ۱۱۴۰ میلی‌ثانیه به ۱۶۵۲ میلی‌ثانیه بدتر شد.

مانع اصلی در ساختار کوئری:

وجود توابع و محاسبات روی ستون‌ها (مثل date_trunc و COALESCE) در بخش‌های GROUP BY و SORT باعث شد ایندکس‌های معمولی روی ستون‌های خام کاربردی نداشته باشند.

اقدام و نتیجه:

تست تمام ایندکس‌های کاندید نه‌تنها به بهبود سرعت کمک نکرد، بلکه به دلیل ماهیت تحلیلی و تجمعی کوئری Q6، عملاً بی‌فایده بود.

هزینه‌های پرداخت‌شده:

در ازای این کار، افت عملکرد شدید (کند شدن کوئری)، هدررفت فضای دیسک برای ایندکس‌های بی‌استفاده و تحمیل سربار اضافی I/O روی دیتابیس را پرداختم.

## ۷. View و جدول snapshot

دو `View` ساخته شده است: `v_delivered_orders` (ساده، فیلتر سفارش‌های تحویل‌شده) و `v_order_items_enriched` (چندجدولی، پیوند سفارش، مشتری، آیتم، محصول و ترجمه‌ی دسته). کوئری ۱ نهایی روی `View` نوشته می‌شود.

برای سنگین‌ترین تحلیل، دو راه را زمان می‌گیرم: `View` که هر بار محاسبه می‌کند و جدول `snapshot` که از پیش محاسبه شده. نتیجه‌ی هر دو باید یکی باشد (تعداد سطر و مجموع فروش).

In [20]:
step_results.append(run_step(engine, "q1", "v2_view", load_query("q1_revenue_view.sql")))
step_results.append(run_step(engine, "q1", "v3_snapshot", load_query("q1_snapshot.sql")))
display(summarize_steps([result for result in step_results if result["query"] == "q1"]))

with engine.connect() as connection:
    view_result = pd.read_sql_query(text(load_query("q1_revenue_view.sql")), connection)
    snapshot_result = pd.read_sql_query(text(load_query("q1_snapshot.sql")), connection)

print("same shape:", view_result.shape == snapshot_result.shape)
print("sales view / snapshot:",
      float(view_result["total_sales"].sum()), float(snapshot_result["total_sales"].sum()))

,query,step,index_name,median_ms,min_ms,max_ms,spread_percent,execution_ms,within_budget,seq_scans,index_used
0,q1,v1_no_index,NaN,2305.51,2055.20,3425.65,59.4,2782.404,False,"category_translation, order_items, orders, products",None
1,q1,v1_with_idx_order_items_product_id,idx_order_items_product_id,1676.26,1426.33,2152.75,43.3,1293.678,False,"category_translation, order_items, orders, products",False
2,q1,v2_view,NaN,1242.75,1133.26,1375.99,19.5,1217.885,False,"category_translation, customers, order_items, orders, products",None
3,q1,v3_snapshot,NaN,27.74,16.79,33.20,59.2,12.758,True,monthly_category_snapshot,None


same shape: True
sales view / snapshot: 13496408.43 13496408.43


In [21]:
q1_view_step = next(
    result for result in step_results if result["query"] == "q1" and result["step"] == "v2_view"
)
q1_final_file = "q1_revenue_view.sql" if q1_view_step["within_budget"] else "q1_snapshot.sql"
print("q1 final uses:", q1_final_file)

FINAL_FILES = {
    "q1": q1_final_file,
    "q2": QUERY_FILES["q2"],
    "q3": QUERY_FILES["q3"],
    "q4": QUERY_FILES["q4"],
    "q5": QUERY_FILES["q5"],
    "q6": "q6_rolling_snapshot.sql",
}
final_results = [
    run_step(engine, query_name, "final", load_query(file_name))
    for query_name, file_name in FINAL_FILES.items()
]
step_results.extend(final_results)

final_table = summarize_steps(final_results)[
    ["query", "median_ms", "min_ms", "max_ms", "execution_ms", "within_budget"]
]
display(final_table)
print("all within the", BUDGET_MS, "ms budget:", bool(final_table["within_budget"].all()))

q1 final uses: q1_snapshot.sql


,query,median_ms,min_ms,max_ms,execution_ms,within_budget
0,q1,28.92,20.70,34.84,28.050,True
1,q2,436.19,381.42,642.13,509.372,False
2,q3,744.80,586.73,994.16,891.382,False
3,q4,1149.90,839.58,1325.06,1178.925,False
4,q5,102.46,54.27,161.52,140.469,True
5,q6,144.61,126.98,221.74,99.815,True


all within the 200 ms budget: False


In [22]:
optimization_trail = summarize_steps(step_results)[
    ["query", "step", "median_ms", "within_budget", "seq_scans", "index_used"]
]
display(optimization_trail)

,query,step,median_ms,within_budget,seq_scans,index_used
0,q1,v1_no_index,2305.51,False,"category_translation, order_items, orders, products",None
1,q2,v1_no_index,673.38,False,"customers, orders",None
2,q3,v1_no_index,1130.81,False,"order_reviews, orders",None
3,q4,v1_no_index,634.72,False,"order_items, order_reviews",None
4,q5,v1_no_index,66.09,True,order_payments,None
5,q6,v1_no_index,1774.19,False,"category_translation, customers, order_items, orders, products",None
6,q1,v1_with_idx_order_items_product_id,1676.26,False,"category_translation, order_items, orders, products",False
7,q2,v1_with_idx_orders_customer_id,383.39,False,"customers, orders",False
8,q2,v1_with_idx_orders_purchase_ts,435.51,False,"customers, orders",False
9,q2,v1_with_all_relevant_indexes,360.18,False,"customers, orders",None


<div dir="rtl">

گزارش انتخاب استراتژی داشبورد
سنگین‌ترین تحلیل (View در برابر Snapshot):

زمان اجرا برای View حدود 1242.75 ms بود، در حالی که همان کوئری با استفاده از Snapshot به 27.74 ms کاهش یافت. این یعنی استفاده از اسنپ‌شات، سرعت اجرا را بیش از ۴۴ برابر بهینه کرد.

انتخاب من برای داشبورد:

استراتژی ترکیبی:

استفاده از Snapshot: برای کوئری‌های سنگینِ تجمیعی (Q1 و Q6) که پیش‌تر خارج از بودجه بودند.
اجرای مستقیم (Raw Query): برای کوئری‌هایی که بدون نیاز به اسنپ‌شات در بودجه قرار گرفتند (مثل Q5).
هزینه‌های پرداخت‌شده:

تازگی داده (Data Freshness): داده‌ها دیگر «لختِ لحظه‌ای» نیستند؛ نیازمند یک فرآیند (Pipeline) برای رفرش دوره‌ای اسنپ‌شات‌ها هستم.
فضا (Space): اشغال فضای دیسکِ اضافه برای نگهداری جدولِ اسنپ‌شات.
پیچیدگی نگهداری (Maintenance): اضافه شدن مسئولیت مدیریتِ فرآیند بروزرسانی (Refresh) اسنپ‌شات‌ها در دیتابیس.


In [23]:
with engine.connect() as connection:
    for file_name in ["q3_delay_vs_review.sql", "q5b_correlation.sql", "q4_top_sellers.sql"]:
        print(file_name)
        display(pd.read_sql_query(text(load_query(file_name)), connection))
    print("q5_payments.sql (first rows)")
    display(pd.read_sql_query(text(load_query("q5_payments.sql")), connection).head(15))
    print("q2_late_delivery.sql (first rows)")
    display(pd.read_sql_query(text(load_query("q2_late_delivery.sql")), connection).head(10))

q3_delay_vs_review.sql


,delay_bucket,orders,avg_review_score
0,1_on_time_or_early,88174,4.294
1,2_late_up_to_3_days,2634,3.764
2,3_late_3_to_7_days,1773,2.317
3,4_late_7_to_14_days,1745,1.744
4,5_late_over_14_days,1504,1.711


q5b_correlation.sql


,installments_value_corr
0,0.3308


q4_top_sellers.sql


,seller_id,revenue,avg_prep_days,avg_review_score
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,2.34,4.13
1,53243585a1d6dc2643021fd1853d8905,222776.05,3.05,4.13
2,4a3ca9315b744ce9f8e9374361493884,200472.92,2.35,3.83
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,2.34,4.34
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,11.37,3.49
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,2.38,4.21
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,2.24,4.18
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1.58,4.24
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,3.74,3.99
9,955fee9216a65b617aa5c0531780ce60,135171.70,1.68,4.16


q5_payments.sql (first rows)


,payment_type,payment_installments,payments,avg_payment_value,share_percent
0,boleto,1,19784,145.03,19.04
1,credit_card,0,2,94.32,0.00
2,credit_card,1,25455,95.87,24.50
3,credit_card,2,12413,127.23,11.95
4,credit_card,3,10461,142.54,10.07
5,credit_card,4,7098,163.98,6.83
6,credit_card,5,5239,183.47,5.04
7,credit_card,6,3920,209.85,3.77
8,credit_card,7,1626,187.67,1.57
9,credit_card,8,4268,307.74,4.11


q2_late_delivery.sql (first rows)


,customer_state,month_start,delivered_orders,late_orders,late_percent
0,AC,2017-01-01,2,0,0.0
1,AC,2017-02-01,3,0,0.0
2,AC,2017-03-01,2,0,0.0
3,AC,2017-04-01,5,0,0.0
4,AC,2017-05-01,8,0,0.0
5,AC,2017-06-01,4,0,0.0
6,AC,2017-07-01,5,0,0.0
7,AC,2017-08-01,4,0,0.0
8,AC,2017-09-01,5,1,20.0
9,AC,2017-10-01,5,0,0.0


<div dir="rtl">

رابطه ی تأخیر و امتیاز (Q3): ⟦رابطه‌ای معکوس و کاملاً محسوس برقرار است؛ به طوری که با افزایش تأخیر در تحویل، امتیاز میانگین مشتریان از ۴.۲۹ برای سفارش‌های به‌موقع، به‌شدت تا ۱.۷۱ برای تأخیرهای طولانی (بیش از ۱۴ روز) سقوط می‌کند.⟧

رابطه‌ی اقساط و ارزش پرداخت (Q5 و ضریب همبستگی): ⟦ضریب همبستگی ۰.۳۳ نشان‌دهنده یک رابطه مثبت متوسط است که تأیید می‌کند خریدهای با ارزشِ ریالی بالاتر، تمایل بیشتری به پرداخت در تعداد اقساطِ بیشتر دارند.⟧

## ۸. Function و Procedure

`olist.delivery_delay_days` یک `Function` است. تعداد روزهای تأخیر را از دو تاریخ حساب می‌کند و داخل `SELECT` و داخل کوئری ۳ به‌کار رفته است.

`olist.rebuild_monthly_snapshot` یک `Procedure` است. با `CALL` اجرا می‌شود، یک ماه را پاک و دوباره می‌سازد. دو ویژگی لازم را دارد:

- **`idempotent`:** اجرای دوباره برای همان ماه همان نتیجه را می‌دهد، چون اول سطرهای آن ماه حذف می‌شوند.
- **رفتار در برابر خطا:** حذف و درج داخل یک `CALL` و یک تراکنش‌اند. اگر وسط کار خطا بیاید، همه برمی‌گردد و `snapshot` قبلی همان ماه سالم می‌ماند.

In [24]:
with engine.connect() as connection:
    delay_example = pd.read_sql_query(
        text(
            "SELECT order_id, "
            "olist.delivery_delay_days(order_delivered_customer_date, order_estimated_delivery_date) "
            "AS delay_days "
            "FROM olist.v_delivered_orders ORDER BY delay_days DESC LIMIT 5"
        ),
        connection,
    )
display(delay_example)

,order_id,delay_days
0,1b3190b2dfa9d789e1f14c05b647a14a,188.98
1,ca07593549f1816d26a572e06dc1eab6,181.61
2,47b40429ed8cce3aee9199792275433f,175.87
3,2fe324febf907e3ea3f2aa9650869fa5,167.71
4,285ab9426d6982034523a855f55a885e,166.58


In [25]:
months = list_purchase_months(engine)
test_month = months[len(months) // 2]
state_sql = text(
    "SELECT COUNT(*), SUM(total_sales) FROM olist.monthly_category_snapshot "
    "WHERE month_start = :month_start"
)

states = []
for _ in range(2):
    with engine.begin() as connection:
        rebuild_month(connection, test_month)
        states.append(tuple(connection.execute(state_sql, {"month_start": test_month}).one()))

print("month:", test_month, "| state after call 1 and call 2:", states)
assert states[0] == states[1], "procedure is not idempotent"

month: 2017-10-01 | state after call 1 and call 2: [(599, Decimal('660179.62')), (599, Decimal('660179.62'))]


In [26]:
with engine.connect() as connection:
    state_before = tuple(connection.execute(state_sql, {"month_start": test_month}).one())

with engine.begin() as connection:
    connection.exec_driver_sql(
        "ALTER TABLE olist.monthly_category_snapshot "
        "ADD CONSTRAINT tmp_block_insert CHECK (total_sales < 0) NOT VALID"
    )

error_name = None
try:
    with engine.begin() as connection:
        rebuild_month(connection, test_month)
except Exception as error:
    error_name = type(error).__name__
finally:
    with engine.begin() as connection:
        connection.exec_driver_sql(
            "ALTER TABLE olist.monthly_category_snapshot DROP CONSTRAINT tmp_block_insert"
        )

with engine.connect() as connection:
    state_after = tuple(connection.execute(state_sql, {"month_start": test_month}).one())

print("error raised:", error_name)
print("state before / after the failed call:", state_before, state_after)
assert state_before == state_after, "failed call left partial data"

error raised: IntegrityError
state before / after the failed call: (599, Decimal('660179.62')) (599, Decimal('660179.62'))


### تفاوت Function و Procedure 

- `delivery_delay_days` باید یک مقدار برگرداند و داخل `SELECT` و کوئری ۳ استفاده شود. فقط `Function` را می‌شود این‌طور به‌کار برد. `Procedure` را نمی‌شود داخل `SELECT` گذاشت. این تابع فقط محاسبه است، پس `IMMUTABLE` تعریف شده و `PostgreSQL` می‌تواند آن را داخل کوئری بازنویسی کند.
- `rebuild_monthly_snapshot` یک کار است نه یک مقدار: پارامتر می‌گیرد، جدول را تغییر می‌دهد و با `CALL` اجرا می‌شود. مزیت مشخص `Procedure` این است که کنترل تراکنش (`COMMIT` و `ROLLBACK` داخل بدنه) را می‌شود در آن نوشت، که در `Function` ممکن نیست. من عمداً داخلش `COMMIT` نگذاشتم تا کل بازسازی یک ماه اتمی بماند. شاهدش آزمایش بالاست که با شکست درج، سطرهای قبلی همان ماه سالم ماندند.

## ۹. پل SQLAlchemy و DataFrame

تمام تعامل با دیتابیس از طریق `SQLAlchemy` است. اتصال‌ها با `with` باز و بسته می‌شوند و در آخر نوتبوک وجود اتصال بازمانده بررسی می‌شود. پارامترهای کوئری همه با `:name` بسته می‌شوند. فقط دو جا متن `SQL` ساخته می‌شود و هیچ مقداری از ورودی در آن نیست: `EXPLAIN` روی متن فایل‌های نسخه‌دار خودم، و `DROP INDEX` که نام `Index` را با `identifier_preparer` امن می‌کند.

خواندن کوئری بزرگ دسته‌ای است (`stream_results` و `chunksize`). نوشتن `DataFrame` با نگاشت نوع ستون‌های صریح انجام می‌شود، نه نوع‌های خودکار. خواندن از `View` هم نشان داده می‌شود.

In [27]:
batch_count = 0
batch_rows = 0
for batch in read_in_batches(
    engine,
    "SELECT order_id, price FROM olist.v_order_items_enriched "
    "WHERE order_purchase_timestamp >= :start_date",
    params={"start_date": "2018-01-01"},
    batch_size=20_000,
):
    batch_count += 1
    batch_rows += len(batch)
print("batches:", batch_count, "| rows read from the view:", batch_rows)

with engine.connect() as connection:
    display(pd.read_sql_query(text("SELECT * FROM olist.v_delivered_orders LIMIT 3"), connection))

batches: 4 | rows read from the view: 61416


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


In [28]:
benchmark_frame = (
    summarize_steps(step_results)[["query", "step", "median_ms"]]
    .rename(columns={"query": "query_name", "step": "variant"})
    .drop_duplicates(["query_name", "variant"], keep="last")
)
benchmark_types = {
    "query_name": types.String(30),
    "variant": types.String(120),
    "median_ms": types.Numeric(10, 2),
}
PostgresDestination(engine).replace_frame(benchmark_frame, "benchmark_results", benchmark_types)

with engine.connect() as connection:
    display(pd.read_sql_query(
        text("SELECT * FROM olist.benchmark_results ORDER BY query_name, median_ms LIMIT 10"),
        connection,
    ))

,query_name,variant,median_ms
0,q1,v3_snapshot,27.74
1,q1,final,28.92
2,q1,v2_view,1242.75
3,q1,v1_with_idx_order_items_product_id,1676.26
4,q1,v1_no_index,2305.51
5,q2,v1_with_all_relevant_indexes,360.18
6,q2,v1_with_idx_orders_customer_id,383.39
7,q2,v1_with_idx_orders_purchase_ts,435.51
8,q2,final,436.19
9,q2,v1_no_index,673.38


## ۱۰. تست‌ها

تست‌های لایه‌ی داده روی دیتابیس جدا (`olist_test`) و با چند سطر کنترل‌شده اجرا می‌شوند: مقدار برگشتی `Function`، `idempotent` بودن `Procedure`، صحت نتیجه‌ی `snapshot`، `View`، رد شدن سطر یتیم و قیمت منفی توسط دیتابیس، و تکرارپذیری `write_clean`. تست‌های قانون‌های قرنطینه بدون دیتابیس اجرا می‌شوند.

In [29]:
completed = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "tests"],
    capture_output=True, text=True,
)
print(completed.stderr[-1500:])
print("return code:", completed.returncode)
assert completed.returncode == 0, "tests failed"

...........
----------------------------------------------------------------------
Ran 11 tests in 1.383s

OK

return code: 0


In [30]:
total_seconds = time.perf_counter() - notebook_started
print("notebook seconds:", round(total_seconds, 1),
      "| excluding the two data loads:", round(total_seconds - load_seconds, 1),
      "| limit: 600")

engine.dispose()
check_engine = make_engine()
with check_engine.connect() as connection:
    open_connections = connection.execute(
        text(
            "SELECT COUNT(*) FROM pg_stat_activity "
            "WHERE application_name = :app_name AND pid <> pg_backend_pid()"
        ),
        {"app_name": APPLICATION_NAME},
    ).scalar_one()
check_engine.dispose()
print("open connections left by the notebook:", open_connections)

notebook seconds: 781.9 | excluding the two data loads: 375.9 | limit: 600
open connections left by the notebook: 0
